In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)


In [2]:
original_df = pd.read_csv("train.csv")
original_df.head()

,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,112163,0,0.00,0,0.0,3,44.500000,0.066667,0.133333,0.000000,0.0,Dec,2,2,9.0,3.0,Returning_Visitor,False,False
1,107490,0,0.00,0,0.0,12,460.200000,0.061111,0.111111,0.000000,0.0,Jul,2,2,6.0,4.0,Returning_Visitor,False,False
2,106273,4,48.80,0,0.0,11,344.800000,0.015385,0.054396,0.000000,0.0,Oct,3,2,1.0,4.0,Returning_Visitor,True,False
3,110651,0,0.00,0,0.0,23,517.035714,0.000000,0.009524,23.300007,0.0,Dec,4,2,8.0,2.0,New_Visitor,False,True
4,101259,7,110.25,0,0.0,20,266.583333,0.011111,0.039753,0.000000,0.0,Mar,2,2,1.0,2.0,Returning_Visitor,False,False


In [3]:
original_df.shape

(9864, 19)

In [7]:
original_df.describe()

,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,9864.000000,9864.000000,9372.000000,9864.000000,9864.000000,9864.000000,9864.000000,9864.000000,9173.000000,9864.000000,9864.000000,9864.000000,9864.000000,9273.000000,9075.000000
mean,106159.495235,2.316910,81.611336,0.500000,33.925878,31.693025,1190.378574,0.022155,0.044035,5.911717,0.061354,2.119019,2.364152,3.169848,4.069421
std,3571.482735,3.346879,181.566335,1.268721,138.415311,44.596694,1943.270384,0.048262,0.049444,18.785453,0.199011,0.910597,1.724511,2.416721,4.027926
min,100001.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,103063.500000,0.000000,0.000000,0.000000,0.000000,7.000000,183.000000,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,106168.500000,1.000000,7.000000,0.000000,0.000000,18.000000,593.725990,0.003080,0.025556,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,109258.750000,4.000000,93.500000,0.000000,0.000000,38.000000,1449.152083,0.016866,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,112330.000000,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


In [8]:
original_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Session_ID               9864 non-null   int64  
 1   Administrative           9864 non-null   int64  
 2   Administrative_Duration  9372 non-null   float64
 3   Informational            9864 non-null   int64  
 4   Informational_Duration   9864 non-null   float64
 5   ProductRelated           9864 non-null   int64  
 6   ProductRelated_Duration  9864 non-null   float64
 7   BounceRates              9864 non-null   float64
 8   ExitRates                9173 non-null   float64
 9   PageValues               9864 non-null   float64
 10  SpecialDay               9864 non-null   float64
 11  Month                    9864 non-null   object 
 12  OperatingSystems         9864 non-null   int64  
 13  Browser                  9864 non-null   int64  
 14  Region                  

In [9]:
original_df.isnull().sum()

Session_ID                   0
Administrative               0
Administrative_Duration    492
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  691
PageValues                   0
SpecialDay                   0
Month                        0
OperatingSystems             0
Browser                      0
Region                     591
TrafficType                789
VisitorType                394
Weekend                      0
Revenue                      0
dtype: int64

In [15]:
original_df['Month'].unique()

array(['Dec', 'Jul', 'Oct', 'Mar', 'Nov', 'Aug', 'May', 'June', 'Feb',
       'Sep'], dtype=object)

In [16]:
#imputer missing numeric values with median
numeric_impute_cols = ["Administrative_Duration", "ExitRates"]
for col in numeric_impute_cols:
    median_val = original_df[col].median()
    original_df[col] = original_df[col].fillna(median_val)
    print(f"Filled {col} missing values with median={median_val:.4f}")

Filled Administrative_Duration missing values with median=7.0000
Filled ExitRates missing values with median=0.0256


In [17]:
#Impute missing categorical/coded values (mode)
categorical_impute_cols = ["Region", "TrafficType", "VisitorType"]
for col in categorical_impute_cols:
    mode_val = original_df[col].mode(dropna=True)[0]
    original_df[col] = original_df[col].fillna(mode_val)
    print(f"Filled {col} missing values with mode={mode_val}")

Filled Region missing values with mode=1.0
Filled TrafficType missing values with mode=2.0
Filled VisitorType missing values with mode=Returning_Visitor


In [18]:
#Region, TrafficType, OperatingSystems, Browser are codes -> integer then category
for col in ["Region", "TrafficType"]:
    original_df[col] = original_df[col].astype(int).astype("category")
 
for col in ["OperatingSystems", "Browser"]:
    original_df[col] = original_df[col].astype("category")

In [19]:
original_df["VisitorType"] = original_df["VisitorType"].astype("category")
original_df["Weekend"] = original_df["Weekend"].astype(bool)
original_df["Revenue"] = original_df["Revenue"].astype(bool)

In [20]:
duration_cols = ["Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]
for col in duration_cols:
    neg_count = (original_df[col] < 0).sum()
    if neg_count > 0:
        print(f"WARNING: {col} has {neg_count} negative values -> clipping to 0")
        original_df[col] = original_df[col].clip(lower=0)
 
for col in ["BounceRates", "ExitRates"]:
    out_of_range = ((original_df[col] < 0) | (original_df[col] > 1)).sum()
    if out_of_range > 0:
        print(f"WARNING: {col} has {out_of_range} values outside [0,1]")

In [22]:
missing_after = original_df.isnull().sum()
missing_after

Session_ID                 0
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [26]:
month_order = ["Jan","Feb","Mar","Apr","May","June","Jul","Aug","Sep","Oct","Nov","Dec"]
original_df["Month"] = pd.Categorical(original_df["Month"], categories=month_order, ordered=True)

In [27]:
processed_df = original_df.drop(columns=['Session_ID']).copy()

In [28]:
nominal_cols = ["VisitorType"]
processed_df = pd.get_dummies(processed_df, columns=nominal_cols, drop_first=True)

In [29]:
# Month: ordered categorical -> integer code (Jan=0 ... Dec=11)
processed_df["Month"] = processed_df["Month"].cat.codes

In [30]:
# Already-numeric-coded categoricals -> plain integers
for col in ["OperatingSystems", "Browser", "Region", "TrafficType"]:
    processed_df[col] = processed_df[col].astype(int)

In [31]:
# Booleans -> int
processed_df["Weekend"] = processed_df["Weekend"].astype(int)
processed_df["Revenue"] = processed_df["Revenue"].astype(int)

In [32]:
# Any one-hot dummy columns are bool -> int
bool_cols = processed_df.select_dtypes(include="bool").columns
processed_df[bool_cols] = processed_df[bool_cols].astype(int)

In [51]:
print("\nprocessed_df shape:", processed_df.shape)
print("processed_df dtypes:")
print(processed_df.dtypes)
print("\nprocessed_df missing values:", processed_df.isnull().sum().sum())


processed_df shape: (9864, 19)
processed_df dtypes:
Administrative                     int64
Administrative_Duration          float64
Informational                      int64
Informational_Duration           float64
ProductRelated                     int64
ProductRelated_Duration          float64
BounceRates                      float64
ExitRates                        float64
PageValues                       float64
SpecialDay                       float64
Month                               int8
OperatingSystems                   int64
Browser                            int64
Region                             int64
TrafficType                        int64
Weekend                            int64
Revenue                            int64
VisitorType_Other                  int64
VisitorType_Returning_Visitor      int64
dtype: object

processed_df missing values: 0


##### Model

In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load your single raw dataset file
# REPLACE 'your_dataset.csv' with the actual name of your single file
full_df = pd.read_csv("train.csv")

# 2. Assign a unique Session_ID if one does not already exist
if "Session_ID" not in full_df.columns:
    full_df["Session_ID"] = np.arange(len(full_df))

# 3. Split into 80% Training and 20% Test sets (stratified by target variable 'Revenue')
original_df, test_df = train_test_split(
    full_df,
    test_size=0.2,
    random_state=42,
    stratify=full_df["Revenue"]
)

# Reset indices to clean up the DataFrames
original_df = original_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# 4. Remove target column 'Revenue' from test_df to match competition conditions
test_df = test_df.drop(columns=["Revenue"])

print("Required Variable 1: original_df shape =", original_df.shape)
print("Required Variable 2: test_df shape     =", test_df.shape)

Required Variable 1: original_df shape = (7891, 19)
Required Variable 2: test_df shape     = (1973, 18)


In [62]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate features and target from original_df
X_train = original_df.drop(columns=["Revenue", "Session_ID"], errors="ignore")
y_train = original_df["Revenue"]

# Define numerical and categorical columns
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "bool"]).columns.tolist()

# Construct preprocessing pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# Fit and transform the training data
X_train_processed = preprocessor.fit_transform(X_train)

# Extract column names after One-Hot Encoding
cat_encoder = preprocessor.named_transformers_["cat"].named_steps["encoder"]
encoded_cat_cols = list(cat_encoder.get_feature_names_out(cat_cols))
all_feature_names = num_cols + encoded_cat_cols

# Required Variable: processed_df
processed_df = pd.DataFrame(X_train_processed, columns=all_feature_names)
print("Required Variable 3: processed_df shape  =", processed_df.shape)

Required Variable 3: processed_df shape  = (7891, 29)


In [63]:
from sklearn.linear_model import LogisticRegression

# Prepare test features
X_test = test_df.drop(columns=["Session_ID"], errors="ignore")
X_test_processed = preprocessor.transform(X_test)
processed_test_df = pd.DataFrame(X_test_processed, columns=all_feature_names)

# Required Variable: model
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model.fit(processed_df, y_train)

# Required Variable: predictions
predictions = model.predict(processed_test_df)

print("Required Variable 4: model fitted successfully.")
print("Required Variable 5: predictions shape  =", predictions.shape)

Required Variable 4: model fitted successfully.
Required Variable 5: predictions shape  = (1973,)


In [64]:
required_vars = ["original_df", "processed_df", "model", "predictions", "test_df"]
missing_vars = [var for var in required_vars if var not in globals()]

if not missing_vars:
    print("✅ All 5 required variables are defined and ready for submission!")
else:
    print(f"❌ Missing variables: {missing_vars}")

✅ All 5 required variables are defined and ready for submission!


In [65]:
# Verify that all 5 required variables exist and are populated
required_vars = ["original_df", "processed_df", "model", "predictions", "test_df"]

for var in required_vars:
    if var in globals():
        print(f"✅ {var} is correctly defined.")
    else:
        print(f"❌ {var} is MISSING!")

✅ original_df is correctly defined.
✅ processed_df is correctly defined.
✅ model is correctly defined.
✅ predictions is correctly defined.
✅ test_df is correctly defined.


In [66]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# Define 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate accuracy, ROC-AUC, Precision, and Recall
cv_results = cross_validate(
    model,
    processed_df,
    y_train,
    cv=cv,
    scoring=['accuracy', 'roc_auc', 'f1']
)

# Print Mean Accuracy & Performance Scores
print(f"Mean CV Accuracy:  {cv_results['test_accuracy'].mean():.4f}")
print(f"Mean CV ROC-AUC:   {cv_results['test_roc_auc'].mean():.4f}")
print(f"Mean CV F1-Score:  {cv_results['test_f1'].mean():.4f}")

Mean CV Accuracy:  0.8465
Mean CV ROC-AUC:   0.8624
Mean CV F1-Score:  0.6044


In [67]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# 1. Split training features into local train and validation sets
X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
    processed_df, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# 2. Fit model on training subset
eval_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
eval_model.fit(X_train_sub, y_train_sub)

# 3. Predict on validation subset
val_preds = eval_model.predict(X_val_sub)
val_probs = eval_model.predict_proba(X_val_sub)[:, 1]

# 4. Display Accuracy Metrics
acc = accuracy_score(y_val_sub, val_preds)
roc = roc_auc_score(y_val_sub, val_probs)

print(f"Validation Accuracy: {acc:.4f} ({acc * 100:.2f}%)")
print(f"Validation ROC-AUC:  {roc:.4f}\n")
print("Detailed Classification Report:")
print(classification_report(y_val_sub, val_preds))

Validation Accuracy: 0.8366 (83.66%)
Validation ROC-AUC:  0.8646

Detailed Classification Report:
              precision    recall  f1-score   support

       False       0.93      0.86      0.90      1314
        True       0.51      0.70      0.59       265

    accuracy                           0.84      1579
   macro avg       0.72      0.78      0.74      1579
weighted avg       0.86      0.84      0.85      1579



In [68]:
import pandas as pd

# 1. Load your new test file
new_test_df = pd.read_csv("test.csv")

# 2. Extract features (drop Session_ID or Target variable if present)
X_new = new_test_df.drop(columns=["Revenue", "Session_ID"], errors="ignore")

# 3. Transform new features using the ALREADY FITTED preprocessor
X_new_processed = preprocessor.transform(X_new)

# Reconstruct DataFrame with feature names (optional, for verification)
processed_new_test_df = pd.DataFrame(X_new_processed, columns=all_feature_names)

# 4. Generate Predictions
new_predictions = model.predict(processed_new_test_df)
new_probabilities = model.predict_proba(processed_new_test_df)[:, 1]

# 5. Attach predictions back to the new data
results_df = new_test_df.copy()
results_df["Predicted_Revenue"] = new_predictions
results_df["Purchase_Probability"] = new_probabilities

print("Predictions generated successfully for new file!")
print(results_df[["Session_ID", "Predicted_Revenue", "Purchase_Probability"]].head())

Predictions generated successfully for new file!
   Session_ID  Predicted_Revenue  Purchase_Probability
0      106094              False              0.220902
1      111845              False              0.289897
2      106794               True              0.961660
3      103444              False              0.190493
4      106833              False              0.250898


In [69]:
# 1. Ensure test features match the preprocessor expected inputs
X_test_final = test_df.drop(columns=["Session_ID", "Revenue"], errors="ignore")

# 2. Transform the current test_df using your fitted preprocessor
X_test_final_processed = preprocessor.transform(X_test_final)

# 3. Regenerate predictions to guarantee length alignment with test_df
predictions = model.predict(X_test_final_processed)

# 4. Verify lengths match before running submission code
print(f"test_df rows:       {len(test_df)}")
print(f"predictions length: {len(predictions)}")

assert len(test_df) == len(predictions), "Mismatch! Fix before generating submission.csv"
print("✅ Lengths match! You can now run the submission cell.")

test_df rows:       1973
predictions length: 1973
✅ Lengths match! You can now run the submission cell.


C:\Users\SOFT\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [70]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Checkpoint information
# ---------------------------------------------------

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (
    processed_rows / original_rows
) * 100


# ---------------------------------------------------
# Detect model used
# ---------------------------------------------------

final_model = model

# Handle GridSearchCV / RandomizedSearchCV
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

# Handle sklearn Pipeline
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__


# ---------------------------------------------------
# Create checkpoint section
# ---------------------------------------------------

checkpoints = pd.DataFrame({

    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})


# ---------------------------------------------------
# Create prediction section
# ---------------------------------------------------

prediction_output = pd.DataFrame({

    "id": test_df["Session_ID"].astype(str),

    "value": np.asarray(predictions).astype(str)

})


# ---------------------------------------------------
# Combine and save
# ---------------------------------------------------

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")
print(checkpoints)


submission.csv created successfully.
                     id               value
0      original_missing                2379
1     processed_missing                   0
2         original_rows                7891
3        processed_rows                7891
4      original_columns                  19
5     processed_columns                  29
6  row_retained_percent               100.0
7            model_name  LogisticRegression
